# 03 - Degrade and augment (CCTV-style, GPU)

Turns the **undegraded** images in `CLEAN_DIR` into the **degraded** training set in
`data/synthetic_degraded/{class}/`, simulating restaurant-camera footage: low resolution, blur,
sensor noise, lighting, colour cast, vignetting, a mounted-camera tilt, and JPEG artifacts. **Novelty #1.**

### Runs on the GPU
The degradation is implemented with **pure PyTorch** (`torch.nn.functional`) and processes images in
**batches on the GPU** (falls back to CPU automatically if no CUDA). No new packages required - it only
uses `torch`, which is already installed. JPEG compression is applied at save time (the file encoder).

### Design notes
- **Class-agnostic & de-biased:** `degrade_batch` never sees the label, so no effect can correlate with
  the class (no "clean = less noise" shortcut). Reads whatever class folders exist on disk (3-class).
- **Per-sample randomness** within a batch for noise / lighting / colour / vignette / geometry; per-batch
  for blur sigma and downscale size.
- **Reflection borders** on the geometric warp (no tell-tale black corners).
- **Reproducible:** one fixed seed for the whole run.
- No timestamp/"REC" HUD is ever drawn (that would be a shortcut cue). Keep both clean + degraded copies
  on disk for the with/without-degradation ablation.

## Configuration + paths

In [ ]:
import os, math, random, shutil
# --- choose the GPU here (no terminal needed; just press Run) ---
# Set BEFORE importing torch. Change "0" if GPU 0 is busy, then restart the kernel.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

torch.set_grad_enabled(False)                       # inference-only preprocessing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- knobs ---
INPUT_DIR  = utils.CLEAN_DIR          # undegraded images, per-class subfolders (e.g. Diff_DataSet)
OUTPUT_DIR = utils.DEGRADED_DIR       # data/synthetic_degraded/{class}/
BATCH = 64                            # images per GPU batch
SIZE = 224                            # working resolution
NUM_AUG_CHOICES = [2, 3, 4, 5]        # degraded copies per source image

CLASSES = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()]) if INPUT_DIR.is_dir() else []
print("device :", DEVICE)
print("input  :", INPUT_DIR)
print("output :", OUTPUT_DIR)
print("classes:", CLASSES)

def reset_degraded_folder():
    if OUTPUT_DIR.exists():
        print("Deleting existing degraded folder..."); shutil.rmtree(OUTPUT_DIR)
    for cls in CLASSES:
        os.makedirs(OUTPUT_DIR / cls, exist_ok=True)
    print("Fresh degraded folder ready:", OUTPUT_DIR)

## GPU degradation (pure PyTorch, batched)

`degrade_batch` takes a batch of images as a `[B,3,H,W]` float tensor in `[0,1]` on the GPU and applies
all CCTV effects with per-sample randomness, returning the degraded batch. JPEG is added later at save.

In [ ]:
def _gauss_kernel(sigma, device, k=5):
    coords = torch.arange(k, device=device, dtype=torch.float32) - (k - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma * sigma)); g = g / g.sum()
    return (g[:, None] * g[None, :]).expand(3, 1, k, k).contiguous()

def _radial_r2(h, w, device):
    yy = torch.linspace(-1, 1, h, device=device)[:, None]
    xx = torch.linspace(-1, 1, w, device=device)[None, :]
    return ((xx ** 2 + yy ** 2) / 2.0).clamp(0, 1)        # normalized r^2 in [0,1]

def degrade_batch(x):
    """x: [B,3,H,W] float in [0,1] on DEVICE -> degraded [B,3,H,W] in [0,1]. Class-agnostic."""
    B, _, H, W = x.shape
    dev = x.device
    def mask(p):                                          # per-sample on/off, shape [B,1,1,1]
        return (torch.rand(B, 1, 1, 1, device=dev) < p).float()

    # 1) Low resolution (the defining CCTV cue) - per-batch size
    s = int(random.choice([112, 128, 160, 192]))
    x = F.interpolate(x, size=(s, s), mode="bilinear", align_corners=False)
    x = F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)

    # 2) Blur / defocus (per-batch sigma)
    if random.random() < 0.6:
        kern = _gauss_kernel(random.uniform(0.3, 2.0), dev)
        x = F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), kern, groups=3)

    # 3) Mounted-camera tilt: per-sample rotation + small translate, reflection border (no black corners)
    ang = (torch.rand(B, device=dev) * 2 - 1) * (15 * math.pi / 180)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = (torch.rand(B, device=dev) * 2 - 1) * 0.06
    theta[:, 1, 2] = (torch.rand(B, device=dev) * 2 - 1) * 0.06
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

    # 4) Sensor noise (per-sample std) - SAME distribution for every class
    std = torch.empty(B, 1, 1, 1, device=dev).uniform_(3, 10) / 255.0
    x = x + torch.randn_like(x) * std * mask(0.6)

    # 5) Lighting: brightness then contrast (per-sample)
    br = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.7, 1.4); mb = mask(0.6)
    x = x * (br * mb + (1 - mb))
    ct = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.7, 1.4); mc = mask(0.6)
    ct = ct * mc + (1 - mc)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - mean) * ct + mean

    # 6) White-balance / colour cast (per-sample, per-channel)
    gain = torch.empty(B, 3, 1, 1, device=dev).uniform_(0.9, 1.12); mcol = mask(0.4)
    x = x * (gain * mcol + (1 - mcol))

    # 7) Vignette / cheap-lens corner darkening (per-sample strength)
    r2 = _radial_r2(H, W, dev)[None, None]
    strg = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.15, 0.4); mv = mask(0.35)
    x = x * (1 - strg * mv * r2)

    return x.clamp(0, 1)

def load_to_tensor(p, size=SIZE):
    img = Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

def save_img(t, path):
    arr = (t.clamp(0, 1) * 255).byte().cpu().numpy().transpose(1, 2, 0)
    q = random.randint(30, 70) if random.random() < 0.6 else 95   # JPEG artifact at save
    Image.fromarray(arr).save(path, quality=q)

## Generate the degraded dataset (batched on the GPU)

2-5 degraded copies per source image; one fixed seed -> reproducible.

In [ ]:
from tqdm import tqdm

def generate_augmented_dataset():
    reset_degraded_folder()
    torch.manual_seed(utils.SEED); random.seed(utils.SEED); np.random.seed(utils.SEED)
    exts = (".jpg", ".jpeg", ".png")
    for cls in CLASSES:
        in_dir, out_dir = INPUT_DIR / cls, OUTPUT_DIR / cls
        srcs = sorted(p for p in in_dir.glob("*") if p.suffix.lower() in exts)
        work = [(p, i) for p in srcs for i in range(random.choice(NUM_AUG_CHOICES))]
        print(f"\n{cls}: {len(srcs)} sources -> {len(work)} degraded")
        for bs in tqdm(range(0, len(work), BATCH)):
            chunk = work[bs:bs + BATCH]
            x = torch.stack([load_to_tensor(p) for p, _ in chunk]).to(DEVICE)
            x = degrade_batch(x)
            for (p, i), t in zip(chunk, x):
                save_img(t, out_dir / f"{p.stem}_aug{i}.jpg")

generate_augmented_dataset()
print("\nDONE.")
for cls in CLASSES:
    print(cls, len(list((OUTPUT_DIR / cls).glob('*'))))

## Balance classes (upsample the smaller classes to the largest)

In [ ]:
def upsample_to_max():
    random.seed(utils.SEED); torch.manual_seed(utils.SEED)
    exts = (".jpg", ".jpeg", ".png")
    counts = {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES}
    target = max(counts.values())
    print("before:", counts, "| target:", target)
    for cls in CLASSES:
        srcs = sorted(p for p in (INPUT_DIR / cls).glob("*") if p.suffix.lower() in exts)
        need = target - counts[cls]
        if need <= 0 or not srcs:
            continue
        picks = [random.choice(srcs) for _ in range(need)]
        for bs in range(0, need, BATCH):
            chunk = picks[bs:bs + BATCH]
            x = degrade_batch(torch.stack([load_to_tensor(p) for p in chunk]).to(DEVICE))
            for j, (p, t) in enumerate(zip(chunk, x)):
                save_img(t, OUTPUT_DIR / cls / f"{p.stem}_extra{bs + j}.jpg")
    print("after :", {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES})

upsample_to_max()

## Visual check: clean (top) vs degraded (bottom) per class

In [ ]:
import matplotlib.pyplot as plt
n = 8
fig, axes = plt.subplots(2 * len(CLASSES), n, figsize=(2 * n, 2 * 2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    clean_files = sorted((INPUT_DIR / cls).glob('*'))[:n]
    for i, cp in enumerate(clean_files):
        axes[2*r, i].imshow(Image.open(cp)); axes[2*r, i].axis('off')
        augs = list((OUTPUT_DIR / cls).glob(f"{cp.stem}_aug*.jpg"))
        ax = axes[2*r+1, i]
        ax.imshow(Image.open(random.choice(augs)) if augs else Image.open(cp)); ax.axis('off')
    axes[2*r, 0].set_ylabel(cls, fontsize=12)
plt.tight_layout(); plt.show()